In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import mean_squared_error

Reading Data and the pre analysis:

In [2]:



# Convert u.item to CSV (optional)

ratings = pd.read_csv('/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/100k/ratings.csv')  # Automatically detects comma separator

movies= pd.read_csv('/Users/chi.nguyenth/Documents/DoAn_63133022_NguyenThiHaChi/dataset/100k/movies.csv')

In [3]:
print(ratings.head())
print(movies.head())

   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  


In [4]:
# Merge để dễ hiển thị
df = pd.merge(ratings, movies, on='movieId')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,1,4.0,964982703,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,1,3,4.0,964981247,Grumpier Old Men (1995),Comedy|Romance
2,1,6,4.0,964982224,Heat (1995),Action|Crime|Thriller
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,5.0,964982931,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [5]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)


In [6]:
# === 3. Create user-item matrix (train only) ===
train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
train_matrix_filled = train_matrix.fillna(0)

# === 4. Compute cosine similarity between items ===
item_sim = cosine_similarity(train_matrix_filled.T)  # Transpose để item-item
item_sim_df = pd.DataFrame(item_sim, index=train_matrix.columns, columns=train_matrix.columns)

Predict function for Item-based

In [7]:
def predict_item_based(user_id, movie_id):
    if user_id not in train_matrix.index or movie_id not in train_matrix.columns:
        return np.nan

    # Các phim user này đã xem
    user_ratings = train_matrix.loc[user_id]
    rated_items = user_ratings[user_ratings.notna()]

    if len(rated_items) == 0:
        return train_matrix.mean().mean()  # fallback: global mean

    sims = item_sim_df.loc[movie_id, rated_items.index]

    if sims.sum() == 0:
        return rated_items.mean()  # fallback: user mean

    weighted_sum = (sims * rated_items).sum()
    sim_sum = sims.sum()

    return weighted_sum / sim_sum

In [8]:
# === 6. Predict on test set ===
test_df = test_df.copy()
test_df['pred_item'] = test_df.apply(lambda row: predict_item_based(row['userId'], row['movieId']), axis=1)

In [9]:
# === 7. Compute RMSE ===
valid_preds_item = test_df.dropna(subset=['pred_item'])
rmse_item = np.sqrt(mean_squared_error(valid_preds_item['rating'], valid_preds_item['pred_item']))
print(f'🎬 Item-based CF RMSE: {rmse_item:.4f}')

🎬 Item-based CF RMSE: 0.9203


User-based Collaborative Filtering (Neighborhood-based)

In [10]:
# === 3. Create user-item matrix (train only) ===
train_matrix = train_df.pivot_table(index='userId', columns='movieId', values='rating')
train_matrix_filled = train_matrix.fillna(0)


In [11]:
# === 4. Compute cosine similarity between users ===
user_sim = cosine_similarity(train_matrix_filled)
user_sim_df = pd.DataFrame(user_sim, index=train_matrix.index, columns=train_matrix.index)

In [12]:
# === 5. Predict function ===
def predict_rating(user_id, movie_id):
    if user_id not in train_matrix.index or movie_id not in train_matrix.columns:
        return np.nan

    sims = user_sim_df.loc[user_id]
    movie_ratings = train_matrix[movie_id]

    # Chỉ lấy user đã đánh giá phim này
    valid_ratings = movie_ratings[movie_ratings.notna()]
    sims = sims[valid_ratings.index]

    if sims.sum() == 0:
        return train_matrix.loc[user_id].mean()  # fallback

    return (valid_ratings * sims).sum() / sims.sum()

In [13]:
# === 6. Predict on test set ===
test_df = test_df.copy()
test_df['pred'] = test_df.apply(lambda row: predict_rating(row['userId'], row['movieId']), axis=1)

In [14]:
# === 7. Compute RMSE ===
valid_preds = test_df.dropna(subset=['pred'])  # Bỏ dòng không dự đoán được
rmse = np.sqrt(mean_squared_error(valid_preds['rating'], valid_preds['pred']))
print(f'✅ RMSE: {rmse:.4f}')

✅ RMSE: 0.9691


In [15]:
from collections import defaultdict

# === 8. Build ground truth từ test set ===
# Các phim user đã thực sự thích (rating >= 4.0)
def get_relevant_items(test_df):
    relevant = defaultdict(set)
    for _, row in test_df.iterrows():
        if row['rating'] >= 4.0:
            relevant[row['userId']].add(row['movieId'])
    return relevant

# === 9. Dự đoán tất cả phim chưa xem + chọn top-N ===
def get_top_n_recommendations(model_type='user', N=10):
    top_n = defaultdict(list)

    for user_id in train_matrix.index:
        user_seen = set(train_matrix.loc[user_id].dropna().index)
        all_movies = set(train_matrix.columns)
        unseen = all_movies - user_seen

        preds = []
        for movie_id in unseen:
            if model_type == 'user':
                pred = predict_rating(user_id, movie_id)
            else:
                pred = predict_item_based(user_id, movie_id)
            if not np.isnan(pred):
                preds.append((movie_id, pred))

        # Lấy top-N phim có rating dự đoán cao nhất
        top_n[user_id] = sorted(preds, key=lambda x: x[1], reverse=True)[:N]

    return top_n

# === 10. Tính Precision@K ===
def precision_at_k(top_n_preds, relevant_items, k=10):
    precisions = []
    for user_id, recs in top_n_preds.items():
        recommended_ids = [movie_id for movie_id, _ in recs]
        if user_id in relevant_items:
            relevant_set = relevant_items[user_id]
            true_positives = len(set(recommended_ids) & relevant_set)
            precision = true_positives / k
            precisions.append(precision)
    return np.mean(precisions)


In [16]:
relevant = get_relevant_items(test_df)

topn_user = get_top_n_recommendations(model_type='user', N=10)
topn_item = get_top_n_recommendations(model_type='item', N=10)

precision_user = precision_at_k(topn_user, relevant, k=10)
precision_item = precision_at_k(topn_item, relevant, k=10)

print(f"🎯 Precision@10 (User-based): {precision_user:.4f}")
print(f"🎯 Precision@10 (Item-based): {precision_item:.4f}")


🎯 Precision@10 (User-based): 0.0003
🎯 Precision@10 (Item-based): 0.0000


SVD


In [17]:
from surprise import Dataset, Reader, SVD
from surprise.model_selection import train_test_split
from surprise import accuracy
from collections import defaultdict

# === 1. Load data từ pandas DataFrame ===
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings[['userId', 'movieId', 'rating']], reader)

# === 2. Split thành train/test ===
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

# === 3. Train mô hình SVD ===
model = SVD()
model.fit(trainset)

# === 4. Dự đoán trên test set ===
predictions = model.test(testset)

# === 5. Tính RMSE ===
rmse_svd = accuracy.rmse(predictions)


RMSE: 0.8818


In [18]:
def get_top_n(predictions, n=10):
    '''Return top-N recommendation for each user from predictions'''
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    
    # Sort and cut off at N
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

def precision_at_k_svd(top_n, threshold=4.0):
    '''Compute precision@k from top_n preds'''
    precisions = []
    for uid, user_ratings in top_n.items():
        n_rel = sum((true_r >= threshold) for (_, true_r) in user_ratings)
        precision = n_rel / len(user_ratings) if user_ratings else 0
        precisions.append(precision)
    return np.mean(precisions)


In [19]:
# Lấy top-N từ predictions
top_n_svd = get_top_n(predictions, n=10)

# Tính precision
precision_svd = precision_at_k_svd(top_n_svd)

print(f"🎯 Precision@10 (SVD): {precision_svd:.4f}")


🎯 Precision@10 (SVD): 0.4508
